# Ensemble coverage and probability-of-exceedance Brier score

This notebook calculates per-case ensemble coverage and probability-of-exceedance Brier scores, then aggregates and maps them. Both metrics retain the forecast initialization `time` and `prediction_timedelta` dimensions. The two-dimensional `verification_time` coordinate records the observation timestamp used for every initialization/lead-time pair.

The synthetic input cell below makes the notebook runnable as-is. Replace it with your forecast and observation `xarray.DataArray` objects when using real data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import cartopy.crs as ccrs

from heatextremes.metrics import (
    coverage,
    mean_in_time_batches,
    probability_of_exceedance_brier_score,
    valid_time_mask,
)
import heatextremes as he 
from dask.diagnostics import ProgressBar 
from dask.distributed import Client


import warnings
from zarr.errors import ZarrUserWarning

temporal_resolution = "daily"  # "daily" or "6-hourly"
variable = "t2m_max_6h"
forecast_days = 15
poe_threshold = 32
coverage_percentile = 90
region_name = "Delhi, India"

# Select an inclusive initialization-time range; leave both as None for all.
initialization_start = None  # e.g. "2020-06-01"
initialization_end = None  # e.g. "2020-08-31"

# For instantaneous 6-hourly skill, set temporal_resolution = "6-hourly",
# variable = "2m_temperature", and target_hour = 0 for 00Z.
# target_hour and target_day_offsets are not meaningful for daily minima, means, or maxima.
target_hour = None
# For example, target_day_offsets = 5 and target_hour = 0 selects 00Z
# on the date five calendar days after *each* initialization.
target_day_offsets = None

# Optional exact forecast horizons. For example, use [np.timedelta64(3, "D")]
# for day-3 only, or [np.timedelta64(3, "D"), np.timedelta64(7, "D")]
# for day-3 and day-7. Leave as None to include every lead before forecast_days.
lead_times = None


In [ ]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(
    n_workers=6,        # match the CPUs you requested
    threads_per_worker=1,
    processes=True,
    memory_limit="3GiB",  # tune to your node memory
)
client = Client(cluster)
client

## Input arrays

The notebook can evaluate daily summaries or instantaneous 6-hourly temperatures for the first 15 forecast days. Daily variables are `t2m_min_6h`, `t2m_mean_6h`, and `t2m_max_6h`. For skill at a valid UTC hour such as 00Z, use the 6-hourly `2m_temperature` setting and `target_hour = 0`. Set `target_day_offsets = 5` to select 00Z on calendar day 5 after every initialization, even when initialization hours differ.

In [5]:
era5 = he.open_cached_era5(chunks = {"time": 244, "latitude": 90, "longitude": 180}) 
era5

<xarray.Dataset> Size: 631GB
Dimensions:                  (time: 37988, latitude: 721, longitude: 1440)
Coordinates:
  * time                     (time) datetime64[ns] 304kB 2000-01-01 ... 2025-...
  * latitude                 (latitude) float64 6kB -90.0 -89.75 ... 89.75 90.0
  * longitude                (longitude) float64 12kB -180.0 -179.8 ... 179.8
Data variables:
    2m_dewpoint_temperature  (time, latitude, longitude) float32 158GB dask.array<chunksize=(244, 90, 180), meta=np.ndarray>
    2m_temperature           (time, latitude, longitude) float32 158GB dask.array<chunksize=(244, 90, 180), meta=np.ndarray>
    surface_pressure         (time, latitude, longitude) float32 158GB dask.array<chunksize=(244, 90, 180), meta=np.ndarray>
    total_precipitation      (time, latitude, longitude) float32 158GB dask.array<chunksize=(244, 90, 180), meta=np.ndarray>
Attributes: (12/16)
    Conventions:                     CF-1.7
    GRIB_centre:                     ecmf
    GRIB_centreDescription:          European Centre for Medium-Range Weather...
    GRIB_edition:                    1
    GRIB_subCentre:                  0
    cache_complete_year:             True
    ...                              ...
    cache_temporal_resolution:       6 hourly
    cache_time_coverage_end:         2000-12-31T18:00:00
    cache_year:                      2000
    history:                         2024-09-02T04:48 GRIB to CDM+CF via cfgr...
    institution:                     European Centre for Medium-Range Weather...
    total_precipitation_definition:  6-hour accumulation ending at each times...

In [6]:
# need this warning nonsense bbecause zarr versions are hard
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=r"Numcodecs codecs are not in the Zarr version 3 specification.*",
        category=ZarrUserWarning,
    )
    model = he.open_aifs_ensv2() 

model

[########################################] | 100% Completed | 129.39 s


<xarray.Dataset> Size: 115TB
Dimensions:                     (time: 2366, number: 26,
                                 prediction_timedelta: 200, latitude: 721,
                                 longitude: 1440, prediction_timedelta_daily: 50)
Coordinates:
  * time                        (time) datetime64[ns] 19kB 2000-01-01 ... 202...
  * number                      (number) int64 208B 0 1 2 3 4 ... 21 22 23 24 25
  * prediction_timedelta        (prediction_timedelta) timedelta64[ns] 2kB 06...
  * latitude                    (latitude) float64 6kB 90.0 89.75 ... -90.0
  * longitude                   (longitude) float64 12kB 0.0 0.25 ... 359.8
  * prediction_timedelta_daily  (prediction_timedelta_daily) timedelta64[ns] 400B ...
Data variables:
    2m_dewpoint_temperature     (time, number, prediction_timedelta, latitude, longitude) float32 51TB dask.array<chunksize=(1, 26, 24, 90, 180), meta=np.ndarray>
    2m_temperature              (time, number, prediction_timedelta, latitude, longitude) float32 51TB dask.array<chunksize=(1, 26, 24, 90, 180), meta=np.ndarray>
    total_precipitation         (time, number, prediction_timedelta_daily, latitude, longitude) float32 13TB dask.array<chunksize=(1, 26, 10, 90, 180), meta=np.ndarray>

In [10]:
# Daily aggregation happens inside the batch scorer below.
# This keeps the full 6-hourly model array out of the metric graph.

## Normalize and align the spatial grid

The metrics use xarray alignment for non-time dimensions. This cell normalizes all longitudes to the conventional half-open `[-180, 180)` range, sorts them, and limits the arrays to their shared spatial grid.

In [11]:
def normalize_longitude(data: xr.Dataset | xr.DataArray) -> xr.Dataset | xr.DataArray:
    """Convert a longitude coordinate from 0--360 to [-180, 180) and sort it."""
    if "longitude" not in data.coords:
        raise ValueError("All inputs must have a longitude coordinate")
    normalized_longitude = (data.longitude + 180) % 360 - 180
    if normalized_longitude.to_index().has_duplicates:
        raise ValueError("Longitude normalization produced duplicate coordinates")
    return data.assign_coords(longitude=normalized_longitude).sortby("longitude")

# Broad rectangular analysis domains: (south, north, west, east).
# Refine these if a study uses a more specific geographic definition.
region_bounds = {
    "Delhi, India": (27.5, 29.5, 76.5, 78.5),
    "Dakar, Senegal": (13.0, 15.5, -18.5, -16.0),
    "Jaipur, India": (25.5, 27.5, 74.5, 77.0),
    "Lagos, Nigeria": (5.0, 8.0, 2.0, 5.0),
    "Addis Ababa, Ethiopia": (7.5, 10.5, 37.5, 40.5),
    "Chicago, IL": (40.0, 43.0, -89.0, -86.0),
    "College Park, MD": (38.0, 40.0, -78.0, -75.5),
    "Bangladesh": (20.0, 27.0, 88.0, 93.0),
    "Europe 2026 heatwave": (35.0, 60.0, -10.0, 35.0),
    "Philippines": (4.0, 22.0, 116.0, 127.0),
    "Senegal": (12.0, 17.5, -18.0, -11.0),
    "Colombia": (-5.0, 13.0, -80.0, -66.0),
    "Rwanda": (-3.0, -1.0, 28.8, 30.9),
    "Pakistan": (23.0, 38.0, 60.0, 78.0),
    "Nigeria": (4.0, 14.5, 2.0, 15.0),
    "Ethiopia": (3.0, 15.0, 32.5, 48.0),
    "Lao PDR": (13.5, 23.0, 100.0, 108.0),
    "Kenya": (-5.0, 5.5, 33.5, 42.0),
    "Peru": (-19.0, 0.5, -82.0, -68.0),
    "Bolivia": (-23.0, -9.0, -70.0, -57.0),
    "Sierra Leone": (6.8, 10.0, -13.5, -10.5),
    "Indonesia": (-11.0, 6.0, 95.0, 141.0),
}

def subset_region(
    data: xr.Dataset | xr.DataArray, bounds: tuple[float, float, float, float]
) -> xr.Dataset | xr.DataArray:
    """Select a region after longitude normalization."""
    south, north, west, east = bounds
    latitude_slice = (
        slice(south, north)
        if data.latitude.values[0] < data.latitude.values[-1]
        else slice(north, south)
    )
    return data.sel(latitude=latitude_slice, longitude=slice(west, east))

if region_name not in region_bounds:
    raise KeyError(f"Unknown region: {region_name}. Choose from {list(region_bounds)}")

model = normalize_longitude(model)
era5 = normalize_longitude(era5)
model = subset_region(model, region_bounds[region_name])
era5 = subset_region(era5, region_bounds[region_name])

if initialization_start is not None or initialization_end is not None:
    model = model.sel(time=slice(initialization_start, initialization_end))
if model.sizes["time"] == 0:
    raise ValueError("No model initializations remain after the requested selection")

if temporal_resolution == "daily":
    if target_hour is not None or target_day_offsets is not None:
        raise ValueError(
            "target_hour and target_day_offsets are only valid for 6-hourly verification"
        )
    era5 = he.daily_era5_aggregates(era5)
elif temporal_resolution != "6-hourly":
    raise ValueError("temporal_resolution must be 'daily' or '6-hourly'")


# Do not align forecast initialization times with observation times: observations
# are matched later at initialization time + lead time by the metric functions.
excluded_dimensions = {"time", "prediction_timedelta", "number"}
model, era5 = xr.align(
    model, era5, join="inner", exclude=excluded_dimensions, copy=False
)

era5_var = era5[variable]


In [12]:
era5_var

<xarray.DataArray 't2m_max_6h' (time: 2366, number: 26,
                                prediction_timedelta: 15, latitude: 721,
                                longitude: 1440)> Size: 4TB
dask.array<getitem, shape=(2366, 26, 15, 721, 1440), dtype=float32, chunksize=(1, 26, 1, 90, 180), chunktype=numpy.ndarray>
Coordinates:
  * time                  (time) datetime64[ns] 19kB 2000-01-01 ... 2025-12-31
  * number                (number) int64 208B 0 1 2 3 4 5 ... 20 21 22 23 24 25
  * prediction_timedelta  (prediction_timedelta) timedelta64[ns] 120B 0 days ...
  * latitude              (latitude) float64 6kB 90.0 89.75 ... -89.75 -90.0
  * longitude             (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8

## Calculate per-case scores

Each output retains `time`, `prediction_timedelta`, `latitude`, and `longitude`; only `number` is reduced. This leaves the choice of aggregation—by lead, initialization, season, or spatial region—to downstream code.

In [13]:
initialization_batch_size = 1

def calculate_scores(model_batch: xr.Dataset) -> xr.Dataset:
    if temporal_resolution == "daily":
        model_var = he.daily_aifs_aggregates(
            model_batch,
            max_days=forecast_days,
        )[variable]
    else:
        model_var = model_batch[variable].where(
            model_batch.prediction_timedelta < np.timedelta64(forecast_days, "D"),
            drop=True,
        )

    if lead_times is not None:
        model_var = model_var.sel(prediction_timedelta=lead_times)

    valid_time_selection = None
    if target_hour is not None or target_day_offsets is not None:
        valid_time_selection = valid_time_mask(
            model_var,
            target_hour=target_hour,
            target_day_offsets=target_day_offsets,
        )

    scores = xr.Dataset(
        {
            "coverage": coverage(model_var, era5_var, percentile=coverage_percentile),
            "brier_score": probability_of_exceedance_brier_score(
                model_var, era5_var, poe_threshold
            ),
        }
    )
    return scores if valid_time_selection is None else scores.where(valid_time_selection)

summaries = mean_in_time_batches(
    model,
    calculate_scores,
    reductions={
        "map": ("time", "prediction_timedelta"),
        "lead": ("time", "latitude", "longitude"),
    },
    batch_size=initialization_batch_size,
)

coverage_map = summaries["map"]["coverage"]
brier_score_map = summaries["map"]["brier_score"]
mean_coverage_by_lead = summaries["lead"]["coverage"]
mean_brier_by_lead = summaries["lead"]["brier_score"]

: 

: 

: 

## Aggregate and map

These maps average over both initialization and lead time. Change `aggregation_dimensions` to preserve one or both dimensions, or use `groupby` for a different diagnostic.

In [ ]:
figure, axes = plt.subplots(
    ncols=2, figsize=(16, 5), subplot_kw={"projection": ccrs.PlateCarree()}
)

coverage_map.plot(
    ax=axes[0],
    x="longitude",
    y="latitude",
  #  transform=ccrs.PlateCarree(),
    cmap="viridis",
    vmin=0,
    vmax=1,
    cbar_kwargs={"label": "90% interval coverage"},
)
brier_score_map.plot(
    ax=axes[1],
    x="longitude",
    y="latitude",
   # transform=ccrs.PlateCarree(),
    cmap="magma_r",
    vmin=0,
    vmax=1,
    cbar_kwargs={"label": "Brier score"},
)

for axis, title in zip(
    axes,
    ("Mean central-90% ensemble coverage", "Mean exceedance Brier score"),
):
    axis.set_global()
    axis.coastlines(linewidth=0.7)
   # gridlines = axis.gridlines(draw_labels=True, linewidth=0.4, color="gray", alpha=0.5)
   # gridlines.top_labels = False
   # gridlines.right_labels = False
    axis.set_title(title)

figure.tight_layout()


## Example: aggregate by lead time

Because the case-wise results still have `time` and `prediction_timedelta`, aggregating over space and initialization produces lead-time diagnostics without recalculating the metrics. When `target_day_offsets` is set, different initialization hours can require different leads, so the notebook skips this lead-time plot to avoid mixing horizons.

In [ ]:
if target_day_offsets is None:
    lead_days = mean_coverage_by_lead.prediction_timedelta / np.timedelta64(1, "D")

    figure, axis = plt.subplots(figsize=(8, 4))
    axis.plot(lead_days, mean_coverage_by_lead, marker="o", label="90% coverage")
    axis.plot(lead_days, mean_brier_by_lead, marker="o", label="Exceedance Brier score")
    axis.set(xlabel="Lead time (days)", ylabel="Mean score", ylim=(0, 1))
    axis.grid()
    axis.legend()
    figure.tight_layout()
else:
    print("Lead-time plot skipped: target_day_offsets can use different leads per initialization.")
